# Patrón de Comportamiento: Chain of Responsibility

## Introducción
El patrón Chain of Responsibility permite pasar una petición a lo largo de una cadena de manejadores hasta que uno de ellos la procese.

## Objetivos
- Comprender cómo desacoplar el emisor de la petición de su receptor.
- Identificar cuándo es útil el patrón Chain of Responsibility.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Sistema de soporte técnico**
Una solicitud de soporte puede ser atendida por diferentes niveles (primer nivel, segundo nivel, etc.) hasta que alguien la resuelva.

**¿Dónde se usa en proyectos reales?**
En sistemas de soporte, validación de formularios, procesamiento de eventos, etc.

## Sin patrón Chain of Responsibility (forma errónea)
El cliente debe conocer y consultar cada manejador manualmente.

In [6]:
def soporte_tecnico(solicitud):
    if solicitud == 'básica':
        print('Resuelta por primer nivel')
    elif solicitud == 'avanzada':
        print('Resuelta por segundo nivel')
    else:
        print('No resuelta')

soporte_tecnico('básica')
soporte_tecnico('avanzada')
soporte_tecnico('especial')

Resuelta por primer nivel
Resuelta por segundo nivel
No resuelta


## Con patrón Chain of Responsibility (forma correcta)
Cada manejador decide si procesa la petición o la pasa al siguiente.

In [2]:
class Manejador:
    def __init__(self, siguiente=None):
        self.siguiente = siguiente
    def manejar(self, solicitud):
        if self.siguiente:
            self.siguiente.manejar(solicitud)

class PrimerNivel(Manejador):
    def manejar(self, solicitud):
        if solicitud == 'básica':
            print('Resuelta por primer nivel')
        else:
            super().manejar(solicitud)

class SegundoNivel(Manejador):
    def manejar(self, solicitud):
        if solicitud == 'avanzada':
            print('Resuelta por segundo nivel')
        else:
            super().manejar(solicitud)

class TercerNivel(Manejador):
    def manejar(self, solicitud):
        if solicitud == 'especial':
            print('Resuelta por tercer nivel')
        else:
            super().manejar(solicitud)

class SinSolucion(Manejador):
    def manejar(self, solicitud):
        print('No resuelta')

cadena = PrimerNivel(SegundoNivel(TercerNivel(SinSolucion())))
cadena.manejar('básica')
cadena.manejar('avanzada')
cadena.manejar('especial')
cadena.manejar('imposible')

Resuelta por primer nivel
Resuelta por segundo nivel
Resuelta por tercer nivel
No resuelta


## UML del patrón Chain of Responsibility
```plantuml
@startuml
class Manejador {
    + manejar(solicitud)
}
Manejador <|-- PrimerNivel
Manejador <|-- SegundoNivel
Manejador <|-- SinSolucion
PrimerNivel --> Manejador
SegundoNivel --> Manejador
SinSolucion --> Manejador
@enduml
```

## Otro ejemplo de la vida real: Pipeline de middlewares HTTP
**Contexto:** frameworks web como Express.js o Django procesan cada petición HTTP a través de una cadena de middlewares: autenticación, rate limiting, logging, y finalmente el handler de la ruta. Cada middleware puede cortar la cadena (ej. rechazar por falta de token) o dejar pasar la petición al siguiente.

### Sin patrón (forma errónea)
Todos los pasos viven en una sola función; agregar o reordenar un middleware implica editar ese único bloque de código.

In [3]:
def manejar_peticion(peticion):
    if not peticion.get('token'):
        print('401: sin autenticación')
        return
    if peticion.get('requests_ultimo_minuto', 0) > 100:
        print('429: demasiadas peticiones')
        return
    print(f"LOG: petición a {peticion['ruta']}")
    print(f"200: procesando {peticion['ruta']}")

manejar_peticion({'token': 'abc', 'requests_ultimo_minuto': 5, 'ruta': '/usuarios'})
manejar_peticion({'token': None, 'ruta': '/usuarios'})

LOG: petición a /usuarios
200: procesando /usuarios
401: sin autenticación


### Con patrón (forma correcta)
Cada middleware es un eslabón independiente que decide si corta la cadena o la deja continuar. Agregar, quitar o reordenar middlewares no toca el código de los demás.

In [4]:
class Middleware:
    def __init__(self, siguiente=None):
        self.siguiente = siguiente
    def manejar(self, peticion):
        if self.siguiente:
            self.siguiente.manejar(peticion)

class AutenticacionMiddleware(Middleware):
    def manejar(self, peticion):
        if not peticion.get('token'):
            print('401: sin autenticación')
            return
        super().manejar(peticion)

class RateLimitMiddleware(Middleware):
    def manejar(self, peticion):
        if peticion.get('requests_ultimo_minuto', 0) > 100:
            print('429: demasiadas peticiones')
            return
        super().manejar(peticion)

class LoggingMiddleware(Middleware):
    def manejar(self, peticion):
        print(f"LOG: petición a {peticion['ruta']}")
        super().manejar(peticion)

class HandlerFinal(Middleware):
    def manejar(self, peticion):
        print(f"200: procesando {peticion['ruta']}")


pipeline = AutenticacionMiddleware(RateLimitMiddleware(LoggingMiddleware(HandlerFinal())))
pipeline.manejar({'token': 'abc', 'requests_ultimo_minuto': 5, 'ruta': '/usuarios'})
pipeline.manejar({'token': None, 'ruta': '/usuarios'})

LOG: petición a /usuarios
200: procesando /usuarios
401: sin autenticación


### UML del ejemplo de middlewares HTTP
```plantuml
@startuml
abstract class Middleware {
    - siguiente: Middleware
    + manejar(peticion)
}
Middleware <|-- AutenticacionMiddleware
Middleware <|-- RateLimitMiddleware
Middleware <|-- LoggingMiddleware
Middleware <|-- HandlerFinal
AutenticacionMiddleware --> Middleware
@enduml
```

### ¿Dónde más se usa Chain of Responsibility?
- **Middlewares HTTP:** exactamente este ejemplo — Express.js, Django, ASP.NET Core encadenan middlewares antes del handler final.
- **Validación de formularios:** una cadena de validadores donde cada uno revisa un aspecto distinto (formato, longitud, unicidad) y detiene la cadena en el primer error.
- **Sistemas de aprobación:** una solicitud de gasto que pasa por jefe directo → gerente → finanzas, deteniéndose en el primer nivel con autoridad suficiente para aprobarla.
- **Manejo de excepciones:** en algunos frameworks, una excepción se pasa por una cadena de manejadores hasta que uno sabe cómo procesarla (handler de errores 404, 500, genérico).
- **Filtros de contenido:** un pipeline de moderación que revisa spam, lenguaje ofensivo y enlaces maliciosos en cadena antes de publicar un comentario.

**Ejercicio de reflexión:** ¿qué pasaría si quisieras que `LoggingMiddleware` se ejecute siempre, incluso cuando `AutenticacionMiddleware` rechaza la petición? ¿Cómo cambiarías el orden de la cadena para lograrlo?

## Actividad
Crea una cadena de validadores para un formulario web donde cada validador revise un campo diferente.

---
## Explicación de conceptos clave
- **Desacoplamiento:** El emisor no necesita conocer el receptor final.
- **Flexibilidad:** Se pueden agregar o quitar manejadores fácilmente.
- **Aplicación en la vida real:** Útil en sistemas de soporte, validación y procesamiento de eventos.

## Conclusión
El patrón Chain of Responsibility es ideal para sistemas donde varias entidades pueden manejar una petición.